In [9]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score

from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

In [10]:
df = pd.read_csv('../Datasets/preprocessed_liver_data.csv')

X = df.drop('Result', axis=1)
y = df['Result']

In [11]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Store out-of-fold predictions
oof_preds = np.zeros((X.shape[0], 3))  # RF, XGB, MLP

# Store fold metrics
acc_scores = []
auc_scores = []
precision_scores = []
recall_scores = []
f1_scores = []

In [12]:
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n--- Fold {fold} ---")

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # SMOTE on training fold only
    smote = SMOTE(random_state=42)
    X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

    # Base models
    rf = RandomForestClassifier(n_estimators=200, random_state=42)
    xgb = XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss',
        random_state=42
    )
    mlp = MLPClassifier(
        hidden_layer_sizes=(64,),
        max_iter=500,
        random_state=42
    )

    # Train
    rf.fit(X_train_bal, y_train_bal)
    xgb.fit(X_train_bal, y_train_bal)
    mlp.fit(X_train_bal, y_train_bal)

    # OOF probabilities
    oof_preds[val_idx, 0] = rf.predict_proba(X_val)[:, 1]
    oof_preds[val_idx, 1] = xgb.predict_proba(X_val)[:, 1]
    oof_preds[val_idx, 2] = mlp.predict_proba(X_val)[:, 1]


--- Fold 1 ---

--- Fold 2 ---

--- Fold 3 ---

--- Fold 4 ---

--- Fold 5 ---

--- Fold 6 ---

--- Fold 7 ---

--- Fold 8 ---

--- Fold 9 ---

--- Fold 10 ---


In [13]:
meta_model = LogisticRegression(
    penalty='l2',
    solver='liblinear',
    random_state=42
)

meta_model.fit(oof_preds, y)

LogisticRegression(random_state=42, solver='liblinear')

In [14]:
stacked_probs = meta_model.predict_proba(oof_preds)[:, 1]
stacked_preds = (stacked_probs >= 0.5).astype(int)

acc = accuracy_score(y, stacked_preds)
auc = roc_auc_score(y, stacked_probs)
prec = precision_score(y, stacked_preds)
rec = recall_score(y, stacked_preds)
f1 = f1_score(y, stacked_preds)

print("\n====== STACKED MODEL RESULTS ======")
print(f"Accuracy  : {acc:.4f}")
print(f"ROC-AUC   : {auc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1-score  : {f1:.4f}")


====== STACKED MODEL RESULTS ======
Accuracy  : 0.9964
ROC-AUC   : 0.9997
Precision : 0.9970
Recall    : 0.9980
F1-score  : 0.9975


# Nested CV : CV evaluation for stacked model (mean ± std)

In [18]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score
import time

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

stack_acc = []
stack_auc = []
stack_prec = []
stack_rec = []
stack_f1 = []

train_times = []
predict_times = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

    print(f"\n--- Stack CV Fold {fold} ---")

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # ---------- Training time ----------
    start_train = time.perf_counter()


--- Stack CV Fold 1 ---

--- Stack CV Fold 2 ---

--- Stack CV Fold 3 ---

--- Stack CV Fold 4 ---

--- Stack CV Fold 5 ---

--- Stack CV Fold 6 ---

--- Stack CV Fold 7 ---

--- Stack CV Fold 8 ---

--- Stack CV Fold 9 ---

--- Stack CV Fold 10 ---


In [19]:
    # Inner CV for stacking
    inner_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    oof_train = np.zeros((X_train.shape[0], 3))

    for tr_idx, val_idx in inner_skf.split(X_train, y_train):

        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr = y_train.iloc[tr_idx]

        smote = SMOTE(random_state=42)
        X_tr_bal, y_tr_bal = smote.fit_resample(X_tr, y_tr)

        rf = RandomForestClassifier(n_estimators=200, random_state=42)
        xgb = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                            subsample=0.8, colsample_bytree=0.8,
                            eval_metric='logloss', random_state=42)
        mlp = MLPClassifier(hidden_layer_sizes=(64,), max_iter=500, random_state=42)

        rf.fit(X_tr_bal, y_tr_bal)
        xgb.fit(X_tr_bal, y_tr_bal)
        mlp.fit(X_tr_bal, y_tr_bal)

        oof_train[val_idx,0] = rf.predict_proba(X_val)[:,1]
        oof_train[val_idx,1] = xgb.predict_proba(X_val)[:,1]
        oof_train[val_idx,2] = mlp.predict_proba(X_val)[:,1]

    # Meta learner
    meta = LogisticRegression(solver='liblinear', random_state=42)
    meta.fit(oof_train, y_train)

    end_train = time.perf_counter()

    train_times.append(end_train - start_train)

    # ---------- Prediction time ----------
    start_pred = time.perf_counter()

    test_meta = np.column_stack([
        rf.predict_proba(X_test)[:,1],
        xgb.predict_proba(X_test)[:,1],
        mlp.predict_proba(X_test)[:,1]
    ])

    test_probs = meta.predict_proba(test_meta)[:,1]
    test_preds = (test_probs >= 0.5).astype(int)

    end_pred = time.perf_counter()
    predict_times.append(end_pred - start_pred)
    
    acc = accuracy_score(y_test, test_preds)
    auc = roc_auc_score(y_test, test_probs)
    prec = precision_score(y_test, test_preds)
    rec = recall_score(y_test, test_preds)
    f1 = f1_score(y_test, test_preds)
    
    stack_acc.append(acc)
    stack_auc.append(auc)
    stack_prec.append(prec)
    stack_rec.append(rec)
    stack_f1.append(f1)
    
    print(f"Accuracy  : {acc:.4f}")
    print(f"ROC-AUC   : {auc:.4f}")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"F1-score  : {f1:.4f}")

Accuracy  : 0.9959
ROC-AUC   : 0.9988
Precision : 0.9978
Recall    : 0.9964
F1-score  : 0.9971


In [20]:
print("\n====== STACKED MODEL (CV RESULTS) ======")

print(f"Accuracy  : {np.mean(stack_acc):.4f} ± {np.std(stack_acc):.4f}")
print(f"ROC-AUC   : {np.mean(stack_auc):.4f} ± {np.std(stack_auc):.4f}")
print(f"Precision : {np.mean(stack_prec):.4f}")
print(f"Recall    : {np.mean(stack_rec):.4f}")
print(f"F1-score  : {np.mean(stack_f1):.4f}")

print("\n====== Stacked Model Timing ======")

print(f"Average Training Time  : {np.mean(train_times):.4f} seconds")
print(f"Average Prediction Time: {np.mean(predict_times):.6f} seconds")


====== STACKED MODEL (CV RESULTS) ======
Accuracy  : 0.9959 ± 0.0000
ROC-AUC   : 0.9988 ± 0.0000
Precision : 0.9978
Recall    : 0.9964
F1-score  : 0.9971

====== Stacked Model Timing ======
Average Training Time  : 129.7146 seconds
Average Prediction Time: 0.131436 seconds
